# TaaS Assurance Evidence Explorer — Colab validation

**Feature branch:** `feature/assurance-evidence-explorer`  
**Pinned product commit:** `84981c1332eed863ad8f8b00c23edc9c4b5d3602`

This notebook launches the actual TaaS Studio API in this Colab runtime, runs the focused regression tests, creates a clearly labelled **test fixture only** using the internal attestation primitive, and verifies it through the authenticated read-only evidence endpoint. The fixture is not a real assessment and must never be represented as production evidence.

Governed MCP invocation remains disabled. This validates evidence-record integrity/freshness plumbing only; it does not validate assessor truth, authorization or production security.

In [ ]:
import os, sys, subprocess, pathlib, time, urllib.request, urllib.error, json, secrets
from IPython.display import display, HTML
REPO_URL = 'https://github.com/msellamiTN/taas-provenance-discovery.git'
TAAS_COMMIT = '84981c1332eed863ad8f8b00c23edc9c4b5d3602'
WORKDIR = pathlib.Path('/content/taas-provenance-discovery')
if not WORKDIR.exists(): subprocess.run(['git','clone',REPO_URL,str(WORKDIR)],check=True)
subprocess.run(['git','fetch','--all'],cwd=WORKDIR,check=True)
subprocess.run(['git','checkout','--detach',TAAS_COMMIT],cwd=WORKDIR,check=True)
actual=subprocess.check_output(['git','rev-parse','HEAD'],cwd=WORKDIR,text=True).strip()
assert actual == TAAS_COMMIT, (actual,TAAS_COMMIT)
print('Pinned revision:',actual)


In [ ]:
install=subprocess.run([sys.executable,'-m','pip','install','-e','.[studio,test]'],cwd=WORKDIR,text=True,capture_output=True)
print(install.stdout[-4000:])
if install.returncode: print(install.stderr[-8000:]); raise RuntimeError('Dependency installation failed')
tests=subprocess.run([sys.executable,'-m','pytest','-q','tests/test_assurance_evidence.py','tests/test_assurance_evidence_api.py'],cwd=WORKDIR,text=True,capture_output=True)
print(tests.stdout)
if tests.returncode: print(tests.stderr[-8000:]); raise RuntimeError('Focused assurance regression tests failed')


## Start the real Studio API

The API listens on `0.0.0.0` for the Colab proxy. Health checks use loopback. Both credentials below are generated only inside this runtime; do not share or commit them.

In [ ]:
APP_PORT=8000
api_token=secrets.token_urlsafe(32)
attestation_key=secrets.token_urlsafe(48)
os.environ['TAAS_STUDIO_API_TOKEN']=api_token
os.environ['TAAS_EVIDENCE_ATTESTATION_KEY']=attestation_key
os.environ['TAAS_EVIDENCE_DB_PATH']='/content/taas-assurance-evidence.sqlite3'
os.environ.pop('TAAS_MODEL_PROFILE',None)
if 'studio_process' in globals() and studio_process.poll() is None: studio_process.terminate(); studio_process.wait(timeout=10)
studio_log=open('/content/taas-assurance-explorer.log','w',buffering=1)
studio_process=subprocess.Popen([sys.executable,'-m','uvicorn','taas.studio.api:app','--host','0.0.0.0','--port',str(APP_PORT)],cwd=WORKDIR,env=os.environ.copy(),stdout=studio_log,stderr=subprocess.STDOUT)
def get_local(path,headers=None,attempts=45):
    last=None
    for _ in range(attempts):
        if studio_process.poll() is not None: studio_log.flush(); raise RuntimeError('Studio exited; inspect /content/taas-assurance-explorer.log')
        try:
            req=urllib.request.Request(f'http://127.0.0.1:{APP_PORT}{path}',headers=headers or {})
            with urllib.request.urlopen(req,timeout=4) as res: return res.status,res.read().decode()
        except urllib.error.HTTPError as exc: return exc.code,exc.read().decode()
        except Exception as exc: last=exc; time.sleep(1)
    raise RuntimeError(f'Studio did not become ready: {last}')
print('Health:',get_local('/health'))
print('Readiness:',get_local('/ready'))
auth={'Authorization':'Bearer '+api_token}
print('Authenticated Agent registry:',get_local('/api/v1/agents',auth)[0])


## Create a test-only evidence fixture

This cell calls the private Python attestation function from inside the notebook solely to seed a controlled fixture. The assessor label explicitly says fixture-only. This is **not** a real tool-schema assessment, not an external assessor, and not evidence of tool safety.

In [ ]:
from taas.studio.assurance_evidence import attest_evidence, store_evidence
binding={'agent_id':'colab-demo','agent_version':'0.0.1','agent_manifest_sha256':'a'*64,'mcp_snapshot_id':'b'*64,'server_id':'fixture','tool_name':'example','tool_definition_sha256':'c'*64,'arguments_sha256':'d'*64,'goal_spec_sha256':'e'*64,'principal':'service:colab-fixture','policy_version':'fixture-policy','policy_sha256':'f'*64}
envelope=attest_evidence(evidence_type='tool_schema_validation',subject_binding=binding,issuer='taas-colab-validation',assessor='colab-fixture-only-not-a-real-assessor',method='test-fixture-not-an-assessment',payload={'fixture':True,'valid':True})
stored=store_evidence(envelope)
evidence_id=stored['evidence_id']
print('Test fixture evidence ID:',evidence_id)
status,body=get_local('/api/v1/assurance/evidence/'+evidence_id,auth)
assert status==200,(status,body)
result=json.loads(body)
assert result['integrity_verified'] is True
assert result['freshness']=='fresh'
assert result['assessor_claim_truth_verified'] is False
assert result['authorization_granted'] is False
assert result['execution_performed'] is False
print(json.dumps({'http_status':status,'evidence_type':result['evidence']['evidence_type'],'integrity_verified':result['integrity_verified'],'freshness':result['freshness'],'assessor_claim_truth_verified':result['assessor_claim_truth_verified'],'authorization_granted':result['authorization_granted'],'execution_performed':result['execution_performed']},indent=2))


## Open the Studio UI

Paste the evidence ID printed above into **Assure → Evidence explorer**. Confirm the summary and JSON details match the API response. Also test a malformed ID and an unknown 64-character ID; neither should display evidence.

In [ ]:
from google.colab import output
proxy_url=output.eval_js(f'google.colab.kernel.proxyPort({APP_PORT})')
display(HTML(f'<p><a href="{proxy_url}" target="_blank" rel="noopener">Open TaaS Studio — Assurance Evidence Explorer</a></p>'))
print('Temporary Studio API token (runtime only):',api_token)
print('Evidence fixture ID:',evidence_id)


## Acceptance checklist

- [ ] Exact commit pin matches the expected SHA.
- [ ] Focused pytest suite passes.
- [ ] `/health` and `/ready` return HTTP 200.
- [ ] Authenticated evidence inspection returns HTTP 200.
- [ ] UI renders evidence details without injecting HTML from record fields.
- [ ] Malformed and unknown IDs fail closed.
- [ ] The UI clearly states that integrity does not prove assessor truth or authorize execution.

Record the Colab runtime type, Python version, test output, HTTP results and any traceback before marking this feature runtime-verified.

In [ ]:
if 'studio_process' in globals() and studio_process.poll() is None: studio_process.terminate(); studio_process.wait(timeout=10)
if 'studio_log' in globals() and not studio_log.closed: studio_log.flush(); studio_log.close()
print('Studio stopped. Runtime-only test credentials are no longer needed.')
